# exp3: detect regions, then a small VLM recognises each crop

Third approach, following the PaddleOCR-VL pattern (layout detector -> crops -> small recogniser):

1. **Detect**: RapidOCR's text detector gives boxes. Only the boxes are used here, not its recognised text.
2. **Group**: boxes on the same visual row are merged into one region, so a label and its value stay together.
3. **Recognise**: each region is cropped and sent to `glm-ocr` (a recogniser, so no `format=` and no instructions).
4. **Parse**: the same provider / reference rules as exp2 run on the recognised text.

The table at the end compares glm-ocr's text with RapidOCR's own text for each region.

In [2]:
from dataclasses import dataclass
from pathlib import Path
import re, time

import cv2
import numpy as np
import pandas as pd
from ollama import chat
from rapidocr import RapidOCR


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError(f"No git repo found above {start}")


REPO_ROOT = find_repo_root()
train_path = REPO_ROOT / "images" / "train"


@dataclass
class Sample:
    path: Path
    img: np.ndarray  # BGR


def load_images(folder: Path, limit: int | None = None) -> list[Sample]:
    samples = []
    for file_path in sorted(Path(folder).resolve().iterdir()):
        if not file_path.is_file():
            continue
        img = cv2.imread(str(file_path))
        if img is None or img.size == 0:
            continue
        samples.append(Sample(file_path.relative_to(REPO_ROOT), img))
        if limit is not None and len(samples) >= limit:
            break
    return samples


samples = load_images(train_path, limit=5)
engine = RapidOCR()

In [3]:
def detect(img: np.ndarray) -> list[dict]:
    """RapidOCR boxes as axis-aligned pixel boxes. `rapid` is kept only for comparison."""
    result = engine(img)
    if result is None or result.boxes is None:
        return []
    dets = []
    for box, txt in zip(result.boxes, result.txts):
        x1, y1 = box.min(axis=0)
        x2, y2 = box.max(axis=0)
        dets.append({"box": [float(x1), float(y1), float(x2), float(y2)], "rapid": txt})
    return dets


def merge_rows(dets: list[dict], tol: float = 0.5) -> list[dict]:
    """Merge boxes whose vertical centre is within tol * height of a row's first box."""
    rows = []
    for d in sorted(dets, key=lambda d: (d["box"][1] + d["box"][3]) / 2):
        yc = (d["box"][1] + d["box"][3]) / 2
        h = d["box"][3] - d["box"][1]
        for r in reversed(rows[-2:]):   # only the latest rows can match
            if abs(yc - r["yc0"]) <= tol * max(h, r["h0"]):
                r["members"].append(d)
                break
        else:
            rows.append({"yc0": yc, "h0": h, "members": [d]})

    regions = []
    for r in rows:
        ms = sorted(r["members"], key=lambda d: d["box"][0])
        boxes = np.array([m["box"] for m in ms])
        regions.append({
            "box": [boxes[:, 0].min(), boxes[:, 1].min(), boxes[:, 2].max(), boxes[:, 3].max()],
            "rapid": " ".join(m["rapid"] for m in ms),
        })
    return regions


def crop(img: np.ndarray, box, pad: int = 6, min_h: int = 64) -> np.ndarray:
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    c = img[max(int(y1) - pad, 0): min(int(y2) + pad, h), max(int(x1) - pad, 0): min(int(x2) + pad, w)]
    if c.shape[0] < min_h:   # tiny crops are hard for the vision encoder: upscale
        s = min_h / c.shape[0]
        c = cv2.resize(c, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    return c


GLM_MODEL = "glm-ocr:q8_0"


def glm_read(c: np.ndarray) -> str:
    ok, buf = cv2.imencode(".png", c)
    r = chat(
        model=GLM_MODEL,
        messages=[{"role": "user", "content": "Text Recognition:", "images": [buf.tobytes()]}],
        keep_alive="30m",
        options={"temperature": 0, "num_predict": 64},   # a single row never needs more; caps loops
    )
    return r.message.content.strip()

In [4]:
def normalize(regions: list[dict], shape) -> list[dict]:
    """Same item shape as exp2's get_ocr_res: {text, box 0-1000}, top-to-bottom."""
    h, w = shape[:2]
    return [{"text": r["glm"], "box": [int(r["box"][0] / w * 1000), int(r["box"][1] / h * 1000),
                                       int(r["box"][2] / w * 1000), int(r["box"][3] / h * 1000)]}
            for r in regions]


per_image = {}   # image_file -> regions (with glm + rapid text)
timing = []

for s in samples:
    t0 = time.perf_counter()
    regions = merge_rows(detect(s.img))
    det_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    for r in regions:
        try:
            r["glm"] = glm_read(crop(s.img, r["box"]))
        except Exception as e:   # e.g. 500 from the server: keep going, fall back to RapidOCR text
            r["glm"] = r["rapid"]
            r["error"] = str(e)[:80]
    rec_s = time.perf_counter() - t0

    per_image[s.path.name] = regions
    timing.append({"image_file": s.path.name, "regions": len(regions), "detect_s": round(det_s, 2),
                   "recognise_s": round(rec_s, 2), "s_per_region": round(rec_s / max(len(regions), 1), 2),
                   "errors": sum("error" in r for r in regions)})

pd.DataFrame(timing)

[INFO] 2026-11-17 09:55:48,674 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-17 09:55:48,736 [RapidOCR] download_file.py:60: File exists and is valid: /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-17 09:55:48,737 [RapidOCR] main.py:63: Using /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-17 09:55:49,289 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-17 09:55:49,292 [RapidOCR] download_file.py:60: File exists and is valid: /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-17 09:55:49,292 [RapidOCR] main.py:63: Using /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-17 09:55:49,377 [RapidOCR] 

,image_file,regions,detect_s,recognise_s,s_per_region,errors
0,001.jpg,24,1.60,15.51,0.65,0
1,002.jpg,12,0.89,3.21,0.27,0
2,003.jpg,19,0.85,4.31,0.23,0
3,004.jpg,13,0.72,3.14,0.24,0
4,005.jpg,25,0.73,4.31,0.17,0


In [5]:
# Provider and reference rules, copied from exp2, run on the glm-ocr text of each region.
PROVIDER_RX = {name: re.compile(p, re.I) for name, p in {
    "GCash": r"g[\s-]?cash", "Maya": r"\bmaya\b|paymaya", "BDO": r"\bbdo\b|banco de oro",
    "BPI": r"\bbpi\b|bank of the philippine islands", "MariBank": r"mari[\s-]?bank",
    "GoTyme": r"go[\s-]?tyme", "Wise": r"\bwise\b"}.items()}
SENDER_ANCHOR = re.compile(r"^\s*(sent\s*via|from)\b", re.I)


def providers_in(text):
    return [n for n, rx in PROVIDER_RX.items() if rx.search(text)]


def detect_provider(items, window=4):
    texts = [i["text"] for i in items]
    for idx, t in enumerate(texts):
        if SENDER_ANCHOR.search(t):
            for u in texts[idx: idx + 1 + window]:
                if (f := providers_in(u)):
                    return f[0], "anchor"
    mentioned = {p for t in texts for p in providers_in(t)}
    return (mentioned.pop(), "unique") if len(mentioned) == 1 else (None, None)


REF_LABEL = re.compile(r"\bref(?:erence)?\.?\s*(?:no\.?|number|id)(?![a-z])", re.I)
OTHER_LABEL = re.compile(r"^\s*(date|amount|fee|total|from|to|sent|transfer|trace|status|time)\b", re.I)
_digits = lambda s: len(re.findall(r"\d", s))
_clean_ref = lambda s: re.sub(r"[^A-Za-z0-9]", "", s).upper()


def detect_reference(items, window=2):
    for idx, it in enumerate(items):
        m = REF_LABEL.search(it["text"])
        if not m:
            continue
        rest = it["text"][m.end():].strip(" :#.-")
        if _digits(rest) >= 4:
            value = rest
            if idx + 1 < len(items):
                nxt = items[idx + 1]
                if re.fullmatch(r"[\d\s-]{4,}", nxt["text"].strip()) and 0 <= nxt["box"][1] - it["box"][3] <= 25:
                    value += nxt["text"]
            return _clean_ref(value), "same-line"
        for nxt in items[idx + 1: idx + 1 + window]:
            t = nxt["text"].strip()
            if OTHER_LABEL.search(t):
                break
            if _digits(t) >= 4:
                return _clean_ref(t), "next-line"
    return None, None


rows = []
for s in samples:
    regions = per_image[s.path.name]
    glm_items = normalize(regions, s.img.shape)
    rapid_items = normalize([{**r, "glm": r["rapid"]} for r in regions], s.img.shape)
    gp, gref = detect_provider(glm_items), detect_reference(glm_items)
    rp, rref = detect_provider(rapid_items), detect_reference(rapid_items)
    rows.append({"image_file": s.path.name,
                 "provider_glm": gp[0], "provider_rapid": rp[0],
                 "ref_glm": gref[0], "ref_rapid": rref[0],
                 "ref_agree": gref[0] == rref[0]})
pd.DataFrame(rows)

,image_file,provider_glm,provider_rapid,ref_glm,ref_rapid,ref_agree
0,001.jpg,BPI,BPI,464438,46443814,False
1,002.jpg,GCash,GCash,8041012MAY202026744628744PM,8041012MAY202026744,False
2,003.jpg,GoTyme,GoTyme,ITO260831040302011,ITO260831040302011,True
3,004.jpg,GCash,GCash,9041620530721JUN072026536PM,9041620530721JUN072026536PM,True
4,005.jpg,MariBank,MariBank,258662,258662,True


In [6]:
# Regions where glm-ocr and RapidOCR disagree (case/space-insensitive): these are the digits worth checking by eye.
norm = lambda t: re.sub(r"\s+", "", t).lower()
diff = [{"image_file": f, "rapid": r["rapid"], "glm": r["glm"]}
        for f, regs in per_image.items() for r in regs if norm(r["rapid"]) != norm(r["glm"])]
print(f"{len(diff)} of {sum(len(v) for v in per_image.values())} regions differ")
pd.DataFrame(diff)

35 of 93 regions differ


,image_file,rapid,glm
0,001.jpg,strollers 2ma.ULIS Firalle,2. MA. HLN\n3. JAN. WOR
1,001.jpg,sing hours. 81,3 More Jin Wong\nAngel Sarah\nManzan
2,001.jpg,closes. 3 Mac RAFAEL wellyn Manzana,3. Moe\n4. RAFEL SARRO\nwelling Manzanar\nBarr...
3,001.jpg,C. pon 4 Barrientos,4 RAFael\nJavellyn Manzan\nBarrientes
4,001.jpg,6567 3:32 MM· AUID MACOR,3:32 PM
5,001.jpg,ECA Transfer successful!,"Transfer successful!\nSaturday, May 23 2026; 0..."
6,001.jpg,"Saturday, May 23 2026; 03:32:11 PM (GM 13","Saturday, May 23 2026; 03:32:11 PM (GM\nConfir..."
7,001.jpg,Confirmation No. 1779521526959,Confirmation No. 1779521526959\nTransaction Ref.
8,001.jpg,Transaction Ref. No. 464438 14,Transaction Ref. No. 464438
9,001.jpg,Sent via BPI 15,Sent via BPI
